# Exp9.1 — A2 Optimization Stability and Representation Failure Decomposition

Aggregation-only notebook for protocol `a2_stability_decomposition_v1`.

In [1]:
from pathlib import Path
import json
import pandas as pd

def find_repo_root(start=Path.cwd()):
    for p in (start, *start.parents):
        if (p / 'scripts').exists() and (p / 'notebooks').exists():
            return p
    raise RuntimeError('Repository root not found')

repo = find_repo_root()
root = repo / 'notebooks' / 'artifacts' / 'experiment_9_1_a2_stability_decomposition' / 'a2_stability_decomposition_v1'
audit = json.loads((root / 'audit.json').read_text())
audit


{'architecture': '234x234',
 'architecture_shifts': [[2, 3, 4], [2, 3, 4]],
 'collapse_train_ba_threshold': 0.7,
 'diagnostic_epochs': [1, 5, 10, 20, 40, 60, 80, 100],
 'diagnostic_samples': 64,
 'experiment_id': 'experiment_9_1_a2_stability_decomposition',
 'factorial': {'cv_mode': 'within_user',
  'expected_runs': 25,
  'folds': [0, 1, 2, 3, 4],
  'model_seeds': [11, 23, 37, 53, 71],
  'seed_fold_decoupled': True},
 'loader_seed_contract': 'exp73._raw_loaders(data, seed, ...)',
 'model_init_seed_contract': "exp73._e2e_pair_seed(seed, 'model_init')",
 'protocol_version': 'a2_stability_decomposition_v1',
 'question': 'Separate A2 optimization instability from fold difficulty and representation loss.',
 'reproduction': {'expected_runs': 3,
  'seeds': [11, 23, 37],
  'split': 'original Exp8/Exp3 fixed cross-user split',
  'test_samples': 146,
  'test_users': ['user_10', 'user_3', 'user_6'],
  'train_samples': 581,
  'train_users': ['user_0',
   'user_1',
   'user_11',
   'user_12',
   'u

## A. Old A2 reproduction

In [2]:
repro = pd.read_csv(root / 'reproduction_runs.csv')
cols = [c for c in ['seed','reference_test_ba','test_balanced_accuracy','delta_vs_reference_test_ba','best_train_ba','collapsed'] if c in repro.columns]
display(repro[cols])


,seed,reference_test_ba,test_balanced_accuracy,delta_vs_reference_test_ba,best_train_ba,collapsed
0,11,0.530632,0.530632,0.000000,0.940769,False
1,23,0.590230,0.596182,0.005952,0.933489,False
2,37,0.566691,0.566691,0.000000,0.938166,False


## B. Fold × seed factorial

Rows are data folds; columns are exact reusable model initializations.

In [3]:
test_ba = pd.read_csv(root / 'test_ba_matrix.csv', index_col=0)
train_ba = pd.read_csv(root / 'best_train_ba_matrix.csv', index_col=0)
collapse = pd.read_csv(root / 'collapse_matrix.csv', index_col=0)
display(test_ba)
display(train_ba)
display(collapse)


,11,23,37,53,71
fold,,,,,
0,0.626738,0.581564,0.558545,0.580925,0.564372
1,0.590963,0.532737,0.658161,0.607676,0.639024
2,0.313871,0.375569,0.385250,0.387127,0.388763
3,0.553496,0.544798,0.511655,0.523407,0.540919
4,0.390522,0.404896,0.424110,0.443945,0.425478


,11,23,37,53,71
fold,,,,,
0,0.929511,0.931438,0.930839,0.927945,0.920812
1,0.938810,0.931244,0.924507,0.919685,0.943462
2,0.528328,0.545852,0.610190,0.607872,0.567490
3,0.924257,0.931614,0.919246,0.919719,0.938669
4,0.611756,0.622628,0.614931,0.662145,0.611557


,11,23,37,53,71
fold,,,,,
0,0,0,0,0,0
1,0,0,0,0,0
2,1,1,1,1,1
3,0,0,0,0,0
4,1,1,1,1,1


## C. Fold effect vs seed effect

In [4]:
fold_summary = pd.read_csv(root / 'fold_effect_summary.csv')
seed_summary = pd.read_csv(root / 'seed_effect_summary.csv')
collapse_summary = json.loads((root / 'collapse_summary.json').read_text())
display(fold_summary)
display(seed_summary)
collapse_summary


,fold,test_balanced_accuracy_mean,test_balanced_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,best_train_ba_mean,best_train_ba_std,l1_whole_count_test_ba_mean,l1_whole_count_test_ba_std,l1_fixed250_test_ba_mean,l1_fixed250_test_ba_std,l2_whole_count_test_ba_mean,l2_whole_count_test_ba_std,l2_fixed250_test_ba_mean,l2_fixed250_test_ba_std,native_to_l2_whole_test_gap_mean,native_to_l2_whole_test_gap_std,native_to_l2_fixed250_test_gap_mean,native_to_l2_fixed250_test_gap_std,collapse_rate
0,0,0.582429,0.026753,0.584889,0.029456,0.928109,0.004295,0.523072,0.018484,0.615755,0.022160,0.592163,0.018019,0.625795,0.034678,0.009734,0.031825,0.043366,0.041270,0.0
1,1,0.605712,0.048496,0.526715,0.023452,0.931542,0.009812,0.586750,0.044381,0.674855,0.032523,0.618319,0.035712,0.688088,0.037492,0.012607,0.038589,0.082376,0.028437,0.0
2,2,0.370116,0.031856,0.424764,0.053894,0.571946,0.036594,0.416765,0.039936,0.559910,0.014553,0.457959,0.046552,0.565340,0.022578,0.087843,0.056632,0.195224,0.027737,1.0
3,3,0.534855,0.016974,0.555187,0.018661,0.926701,0.008332,0.532674,0.035572,0.625887,0.028048,0.564701,0.025399,0.652267,0.032550,0.029846,0.017287,0.117412,0.035792,0.0
4,4,0.417790,0.020572,0.513061,0.019865,0.624603,0.021460,0.487151,0.008397,0.563728,0.039159,0.498380,0.032999,0.625296,0.030351,0.080590,0.016024,0.207506,0.033809,1.0


,seed,test_balanced_accuracy_mean,test_balanced_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,best_train_ba_mean,best_train_ba_std,l1_whole_count_test_ba_mean,l1_whole_count_test_ba_std,l1_fixed250_test_ba_mean,l1_fixed250_test_ba_std,l2_whole_count_test_ba_mean,l2_whole_count_test_ba_std,l2_fixed250_test_ba_mean,l2_fixed250_test_ba_std,native_to_l2_whole_test_gap_mean,native_to_l2_whole_test_gap_std,native_to_l2_fixed250_test_gap_mean,native_to_l2_fixed250_test_gap_std,collapse_rate
0,11,0.495118,0.135747,0.496457,0.090414,0.786532,0.199885,0.517195,0.084038,0.631286,0.044183,0.552123,0.081062,0.630718,0.049805,0.057004,0.061734,0.135600,0.101036,0.4
1,23,0.487913,0.091554,0.519469,0.080680,0.792555,0.192092,0.499178,0.054483,0.596095,0.036631,0.521039,0.065114,0.591317,0.044449,0.033126,0.034104,0.103405,0.076932,0.4
2,37,0.507544,0.108651,0.538700,0.029631,0.799943,0.171113,0.534200,0.062180,0.596823,0.054337,0.564565,0.069470,0.629842,0.053030,0.057021,0.052724,0.122298,0.059653,0.4
3,53,0.508616,0.092470,0.521789,0.042856,0.807473,0.158639,0.514712,0.065241,0.607948,0.045073,0.556551,0.061157,0.650040,0.055689,0.047935,0.043403,0.141424,0.059794,0.4
4,71,0.511711,0.102941,0.528200,0.068652,0.796398,0.189679,0.481127,0.069381,0.607981,0.078705,0.537244,0.082430,0.654868,0.041187,0.025533,0.051346,0.143157,0.079432,0.4


{'collapse_rate': 0.4,
 'n_successful': 15,
 'n_total': 25,
 'successful_test_ba_mean': 0.5743320457110447,
 'successful_test_ba_std': 0.04347909543411365}

## D. Native A2 vs frozen L1/L2 probes

Use this table to distinguish optimization failure, L2 information loss and readout bottlenecks.

In [5]:
probes = pd.read_csv(root / 'native_vs_probe.csv')
display(probes.sort_values(['collapsed','fold','seed']))


,fold,seed,collapsed,test_balanced_accuracy,l1_whole_count_test_ba,l1_fixed250_test_ba,l2_whole_count_test_ba,l2_fixed250_test_ba,native_to_l2_whole_test_gap,native_to_l2_fixed250_test_gap,l2_minus_l1_whole_test,l2_minus_l1_fixed250_test
0,0,11,False,0.626738,0.516369,0.647131,0.604212,0.649533,-0.022526,0.022794,0.087843,0.002402
1,0,23,False,0.581564,0.552259,0.590171,0.571631,0.565217,-0.009933,-0.016347,0.019372,-0.024953
2,0,37,False,0.558545,0.527834,0.622067,0.599989,0.638802,0.041444,0.080257,0.072155,0.016735
3,0,53,False,0.580925,0.503287,0.620004,0.574182,0.629834,-0.006744,0.048909,0.070895,0.009831
4,0,71,False,0.564372,0.515610,0.599402,0.610803,0.645587,0.046432,0.081215,0.095194,0.046185
5,1,11,False,0.590963,0.618751,0.663684,0.623874,0.685218,0.032911,0.094255,0.005123,0.021534
6,1,23,False,0.532737,0.525763,0.650871,0.579364,0.628089,0.046627,0.095352,0.053600,-0.022783
7,1,37,False,0.658161,0.636879,0.672305,0.666381,0.714849,0.008221,0.056689,0.029502,0.042544
8,1,53,False,0.607676,0.590302,0.656198,0.634816,0.724121,0.027140,0.116445,0.044514,0.067923
9,1,71,False,0.639024,0.562057,0.731215,0.587161,0.688163,-0.051862,0.049139,0.025104,-0.043052


## E. Optimization trajectory diagnostics

In [6]:
training = pd.read_csv(root / 'training_diagnostics.csv')
activity = pd.read_csv(root / 'activity_diagnostics.csv')
display(training.groupby(['fold','seed'])[['train_ba','val_ba','grad_l1','grad_l2','grad_out']].tail(1))
display(activity.head())


,train_ba,val_ba,grad_l1,grad_l2,grad_out
99,0.929511,0.571572,0.016071,0.096122,0.167064
199,0.931438,0.608186,0.018256,0.096480,0.166254
299,0.930839,0.533182,0.017258,0.095128,0.167659
399,0.927945,0.553002,0.016056,0.125295,0.168211
499,0.920812,0.544408,0.014749,0.088430,0.151459
599,0.930460,0.490397,0.015891,0.104631,0.170350
699,0.931244,0.523203,0.018202,0.106486,0.186349
799,0.924507,0.542026,0.018220,0.101188,0.179847
899,0.919685,0.498684,0.017833,0.114556,0.185420
999,0.943462,0.519252,0.014935,0.089135,0.140240


,fold,seed,epoch,layer,shift,mean_spikes_per_neuron_s,dead_neuron_fraction,spike_occupancy
0,0,11,1,l1,all,0.268881,0.437500,0.004201
1,0,11,1,l1,2,0.009186,0.581395,0.000144
2,0,11,1,l1,3,0.067445,0.441860,0.001054
3,0,11,1,l1,4,0.740992,0.285714,0.011578
4,0,11,1,l2,all,2.283823,0.195312,0.035685


## Interpretation guide

- low train BA + low L1/L2 probes → backbone optimization failure;
- low native BA + high L2 probe → native objective/readout failure;
- high L1 probe + low L2 probe → L2 destroys useful representation;
- high train BA + low held-out BA → representation overfit;
- failures concentrated by seed → initialization sensitivity;
- failures concentrated by fold → subset difficulty;
- isolated fold×seed failures → nonlinear interaction.